# Fitting wells

This notebook introduces the concept of fitting wells and how they are used in `gwrefpy`. The methodology originates from the description by its author [Strandanger (2024)](https://svenskageotekniskaforeningen.se/wp-content/uploads/Publikationer/SGF_Rapporter/2024_2_Akvifars_refmetod.pdf).

This notebook can be downloaded from the source code [here](https://github.com/sgfsweden/gwrefpy/blob/main/docs/user_guide/2_fitting_wells.ipynb).

To analyse deviations in groundwater level timeseries, the `gwrefpy` methodology relies on observation wells and reference wells. To check for deviations in an observation well, we fit its data to a reference well using regression. When fitting the two wells, we specify a calibration period. If data outside of the calibration period does not follow the fitted regression, a deviation has occurred.

In this notebook, we will dive deeper into the fitting workflow and showcase this library's associated capabilities.

```{note}
If you are new to `gwrefpy`, start with the [introduction notebook](1_intro_gwrefpy.ipynb), which covers wells and models.
```


In [ ]:
import gwrefpy as gr
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
gr.set_log_level("ERROR")
plt.set_loglevel("ERROR")

## Data

In this notebook, we will work with example data that ships with `gwrefpy`. The example models can be loaded by their file name. Let's load one and plot the observation well and the reference well separately. A `Well` object can be plotted using its `.plot()` method, which shows the timeseries as a line chart.

In [ ]:
model = gr.Model("deviation_example.gwref")
model

In [ ]:
fig, (up, down) = plt.subplots(nrows=2, sharex=True, layout="constrained")

for well in model.wells.obs():
    well.plot(ax=up)
for well in model.wells.ref():
    well.plot(ax=down)

up.set_title("Observation well")
down.set_title("Reference well");

## Anatomy of a fit

Let's walk through the anatomy of a fit by fitting the observation well _19W110U_ to the reference well _19W100U_. To fit wells within a model, we call `model.fit()` which has the following signature:

In [ ]:
help(model.fit)

The first two arguments determine which wells to use.

The `offset` and `aggregation` arguments handle cases where the timeseries of the wells are not aligned, see the [time offsets notebook](3_time_offsets.ipynb). The wells in this model are sampled daily and cover the same period, which means we can use `"0D"` as the offset. With no offset, there is nothing to aggregate, so the `aggregation` argument has no effect.

The `p` argument sets the confidence level of the prediction interval. Observations that fall outside the interval are flagged as deviations. We will use the default, `p=0.95`, i.e. a 95% prediction interval.

The `method` argument decides which regression method to apply. A (simple) linear regression, `Linear()`, is applied by default. See the [fitting methods notebook](7_fitting_methods.ipynb) for polynomial fits.

The `tmin` and `tmax` arguments control the calibration period, i.e. the period whose data is used to fit the regression. This is an important decision that can only be inferred from prior knowledge of the wells. We want to select this period such that no (or negligible) external factors are affecting head in the observation well. In this example, we will assume that data up to and including 2022 is unaffected and can be used for fitting a regression. A partial date like `tmax="2022"` includes the whole year.

```{important}
The reference well needs to be unaffected by external factors during its entire time period.
```

Let's do this!

In [ ]:
fit = model.fit(
    obs_well="19W110U",
    ref_well="19W100U",
    offset="0D",
    tmax="2022",
)
fit

`fit()` returns a `FitResultData` object, which shows a summary of the fitted regression, as seen above.

Let's plot the resulting fit to verify everything is OK. The `plots.diagnostics()` method draws a cross-plot of the two wells: reference well heads on the x-axis and observation well heads on the y-axis. Filled points are inside the calibration period and hollow points are outside it. The solid line is the regression and the dashed lines are the prediction interval.

In [ ]:
fit.plots.diagnostics();

For the calibration data (the filled points), our fit seems OK. Observed heads outside of this period (the hollow points) show a different behavior - one might suspect a deviation!

## Analysing deviations

Evaluating deviations is a key `gwrefpy` feature! Below, we plot the observed heads together with the fitted heads, i.e. the reference well heads transformed by the regression, and the 95% prediction interval around them. With `calibration_period=True`, the calibration period is shaded.

In [ ]:
fit.plot(calibration_period=True);

Seems like the observation well (19W110U) dips below the lower bound during 2023 and 2024. Let's zoom into this period using the `tmin` and `tmax` arguments, which every plotting method accepts. We can also pass `outliers=True` to mark all observation well points that fall outside the prediction interval.

In [ ]:
fit.plot(tmin="2023", tmax="2024", outliers=True);

## Generate multiple fits at once

With `gwrefpy`, you can fit as many reference wells as you like to a single observation well. This is useful if you have many wells to choose from, and would like to rank their fit performance.

Below, we load an example model with many observation and reference wells and clear its existing fits.

In [ ]:
large_model = gr.Model("large_example.gwref")
large_model.fits.clear()

print(
    f"There are {len(large_model.wells)} wells in this model "
    f"and, since just recently, {len(large_model.fits)} fits."
)

We fit the observation well _01OBS_ against every reference well by passing `large_model.wells.ref()` as the reference well. When one side is a single well and the other is a list or collection of wells, `fit()` pairs the single well with each well on the other side. The new fits are stored in the model and also returned as a `FitCollection`.

A `FitCollection` can rank its fits by RMSE: `best()` returns the single best fit and `top(n)` returns the `n` best fits as a new collection. Displaying a collection shows a summary table.

In [ ]:
fits = large_model.fit("01OBS", large_model.wells.ref(), offset="0D", report=False)

top_three = fits.top(3)
top_three

To compare the candidates visually, pass them to `gr.plots.compare()`. It draws the fits in one cross-plot, each fit in its own colour and marker, with its RMSE and R² in the legend. Like every plot, it returns the matplotlib Axes, which we use here to move the legend outside the plot.

In [ ]:
ax = gr.plots.compare(top_three)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5));

Here, _01REF_ follows _01OBS_ most closely: its points scatter least around the regression line, and it has the lowest RMSE and the highest R². This is a great way to evaluate many reference wells for a single observation well!

All twelve fits are still stored in the model, in `large_model.fits`. To keep only the best ones, `large_model.fits.keep_best(3, by="obs")` removes the rest.

## Next steps

- [Time offsets](3_time_offsets.ipynb): fit wells that are not measured at the same times.
- [Plotting](4_plotting.ipynb): all the plots for wells and fits.
- [Fitting methods](7_fitting_methods.ipynb): fit with polynomials instead of a straight line.

Happy fitting!